In [77]:
import os
import pandas as pd
import numpy as np
import random
import dendropy
import treeswift
from treeswift import read_tree_newick
#import matplotlib.pyplot as plt
import re
import copy

In [160]:
path = "cafe_dist"
#phylip_mtrx = "results.D2star.phylip"
#phylip_mtrx = "results.D2shepp.phylip"
#phylip_mtrx = "results.CVtree.phylip"
#phylip_mtrx = "results.Cosine.phylip"

#phylip_mtrx = "results.Co-phylog.phylip"
#phylip_mtrx = "results.Eu.phylip"
#phylip_mtrx = "results.JS.phylip"
phylip_mtrx = "results.Ma.phylip"

dist_name = phylip_mtrx.split(".")[1]

# Read phylip matrix
def read_phylip_dist(filename):
    with open(filename) as f:
        # first line = number of taxa (ignore or check consistency)
        n = int(f.readline().strip())
        labels = []
        matrix = []
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            labels.append(parts[0])         # first entry = sequence ID
            matrix.append([float(x) for x in parts[1:]])  # rest = distances

    # Create square DataFrame
    df = pd.DataFrame(matrix, index=labels, columns=labels)
    return df

# Example usage
df = read_phylip_dist(os.path.join(path, phylip_mtrx))
print(df.head())

                                   G000430525.part_NZ_AUME01000007.1  \
G000430525.part_NZ_AUME01000007.1                           0.000000   
G000424365.part_NZ_AUJT01000008.1                           0.668990   
G000336655.part_NZ_AOLZ01000076.1                           1.086680   
G000163775.part_NZ_JH815302.1                               0.707182   
G000430745.part_NZ_KE387231.1                               0.645879   

                                   G000424365.part_NZ_AUJT01000008.1  \
G000430525.part_NZ_AUME01000007.1                           0.668990   
G000424365.part_NZ_AUJT01000008.1                           0.000000   
G000336655.part_NZ_AOLZ01000076.1                           1.209720   
G000163775.part_NZ_JH815302.1                               0.836860   
G000430745.part_NZ_KE387231.1                               0.702099   

                                   G000336655.part_NZ_AOLZ01000076.1  \
G000430525.part_NZ_AUME01000007.1                            1

In [161]:
df.head(n=5)

,G000430525.part_NZ_AUME01000007.1,G000424365.part_NZ_AUJT01000008.1,G000336655.part_NZ_AOLZ01000076.1,G000163775.part_NZ_JH815302.1,G000430745.part_NZ_KE387231.1,G001894865.part_NZ_BCXA01000020.1,G900110905.part_FOFH01000009.1,G000300115.part_NZ_JH930378.1,G000172095.part_NZ_ABID01000003.1,G000430745.part_NZ_AUMP01000016.1,...,G900097105.part_NZ_LT629973.1,G001313205.part_BBFK01000002.1,G000336655.part_NZ_AOLZ01000022.1,G001005215.part_NZ_LBNQ01000037.1,G000497755.part_NZ_AYOD01000011.1,G000519205.part_NZ_JAFB01000009.1,G000224335.part_NZ_AFXZ01000019.1,G001308105.part_NZ_CP012851.1,G000425585.part_NZ_AUDS01000008.1,G000719275.part_NZ_JOFP01000004.1
G000430525.part_NZ_AUME01000007.1,0.000000,0.668990,1.08668,0.707182,0.645879,1.219140,0.718749,0.675984,0.779657,0.677377,...,0.702490,0.613291,1.144390,0.939469,1.018830,1.130090,0.770614,0.571137,0.547484,1.180070
G000424365.part_NZ_AUJT01000008.1,0.668990,0.000000,1.20972,0.836860,0.702099,1.352250,0.758752,0.729631,0.959522,0.640955,...,0.870620,0.720584,1.260770,1.118700,1.170270,1.277000,0.749233,0.583570,0.604381,1.316320
G000336655.part_NZ_AOLZ01000076.1,1.086680,1.209720,0.00000,1.295960,1.186890,0.592964,1.189020,1.270690,0.809674,1.182270,...,0.748465,1.060950,0.274003,0.846206,0.632678,0.576517,1.390120,1.168310,1.000920,0.607676
G000163775.part_NZ_JH815302.1,0.707182,0.836860,1.29596,0.000000,0.662774,1.395920,0.599773,0.651289,0.943348,0.680690,...,0.925638,0.657512,1.339780,1.099430,1.220910,1.307810,0.641692,0.597201,0.645458,1.348670
G000430745.part_NZ_KE387231.1,0.645879,0.702099,1.18689,0.662774,0.000000,1.333810,0.570130,0.650851,0.911134,0.497858,...,0.829794,0.630837,1.235000,1.073260,1.120040,1.243270,0.586376,0.456090,0.555781,1.294500


In [162]:
if df.isna().values.any():
    print("NaN values found in distance matrix!")

In [163]:
# Find indices where df has NaN
nan_locs = np.where(df.isna())

# Collect sequence ID pairs
nan_pairs = [(df.index[i], df.columns[j]) for i, j in zip(*nan_locs)]

if nan_pairs:
    print("NaN distances found between:")
    for a, b in nan_pairs:
        print(f"{a}  <->  {b}")
else:
    print("No NaN distances in the matrix.")

No NaN distances in the matrix.


In [164]:
df_clean = df.dropna(axis=0, how="any").dropna(axis=1, how="any")

In [165]:
df = copy.deepcopy(df_clean)

In [166]:
# Optional: save to CSV/TSV
df.to_csv(os.path.join(path, "pairwise_distances_cafe_{}_phylip.tsv".format(dist_name.lower())), sep="\t")
#df.to_csv(os.path.join(path, "pairwise_distances_cafe_d2shepp_phylip.tsv"), sep="\t")

In [167]:
import pandas as pd
from itertools import combinations, product

# df is your pairwise distance matrix DataFrame
# Leaves labeled like "sample1.part_001", "sample2.part_010", etc.

# --- Step 1: group contigs by sample ID ---
sample_to_contigs = {}
for leaf in df.index:
    sample_id = leaf.split(".part")[0]  # extract sample ID
    sample_to_contigs.setdefault(sample_id, []).append(leaf)

# --- Step 2: compute within-sample distances ---
within_distances = []
records = []
for sample, contigs in sample_to_contigs.items():
    if len(contigs) < 2:
        continue  # no pairs to compute
    # all pairwise combinations of contigs in the same sample
    for a, b in combinations(contigs, 2):
        within_distances.append(df.loc[a, b])
        records.append({
            'Contig1': a,
            'Contig2': b,
            'Distance': df.loc[a, b],
            'Type': 'Within',
            'Sample1': sample,
            'Sample2': sample
        })

mean_within = sum(within_distances) / len(within_distances)
print("Mean within-sample distance:", mean_within)

# --- Step 3: compute across-sample distances ---
across_distances = []
samples = list(sample_to_contigs.keys())
for i in range(len(samples)):
    for j in range(i + 1, len(samples)):
        contigs_i = sample_to_contigs[samples[i]]
        contigs_j = sample_to_contigs[samples[j]]
        # all pairs between two different samples
        for a, b in product(contigs_i, contigs_j):
            across_distances.append(df.loc[a, b])
            records.append({
                'Contig1': a,
                'Contig2': b,
                'Distance': df.loc[a, b],
                'Type': 'Across',
                'Sample1': samples[i],
                'Sample2': samples[j]
            })

mean_across = sum(across_distances) / len(across_distances)
print("Mean across-sample distance:", mean_across)

# --- Step 4: save to text file ---
output_df = pd.DataFrame(records)

all_distances = output_df['Distance'].values
min_val = all_distances.min()
max_val = all_distances.max()
output_df['Distance_scaled'] = (output_df['Distance'] - min_val) / (max_val - min_val)
output_df['Condition'] ='cafe_phylip_{}'.format(dist_name.lower())


output_df.to_csv(os.path.join(path, "pairwise_distances_cafe_{}_phylip_R.txt".format(dist_name.lower())), sep='\t', index=False)
print("Saved distances")

Mean within-sample distance: 0.35080464179785087
Mean across-sample distance: 0.8898270795746268
Saved distances


In [56]:
0.345/0.097

3.556701030927835

In [9]:
max_val

nan

In [36]:
mean_across/mean_within

3.2316239322737905

In [37]:
# To scale distances relative to the combined range
import numpy as np

# Convert to numpy arrays if not already
within = np.array(within_distances)
across = np.array(across_distances)

# Combine to get the global min and max
all_distances = np.concatenate([within, across])
min_val = all_distances.min()
max_val = all_distances.max()

# Scale each set relative to the combined range
within_scaled = (within - min_val) / (max_val - min_val)
across_scaled = (across - min_val) / (max_val - min_val)

# Example: check min/max after scaling
print("Within scaled:", within_scaled.min(), "-", within_scaled.max())
print("Across scaled:", across_scaled.min(), "-", across_scaled.max())


Within scaled: 0.0 - 0.8217103006440996
Across scaled: 0.0 - 1.0


In [38]:
mean_scaled_within = sum(within_scaled) / len(within_scaled)
print("Mean within-sample distance:", mean_scaled_within)
mean_scaled_across = sum(across_scaled) / len(across_scaled)
print("Mean across-sample distance:", mean_scaled_across)

Mean within-sample distance: 0.03433655863462039
Mean across-sample distance: 0.1109628446347678


In [39]:
mean_scaled_across/mean_scaled_within

3.231623932250675

3.2316239322737905

In [28]:
from scipy.stats import f_oneway

# within_distances and across_distances are your lists of pairwise distances
F_stat, p_value = f_oneway(within_distances, across_distances)

print("F-statistic:", F_stat)
print("p-value:", p_value)

F-statistic: 7820.833961820503
p-value: 0.0


In [11]:
# Save to CSV/TSV
#matrix.to_csv("pairwise_distances.tsv", sep="\t")

#print("Distance matrix saved. Shape:", matrix.shape)